# 06 · Bronze · ERP (canal loja física)
Ingere os arquivos do ERP sintético das 12 lojas físicas da Lojas Horizonte.

- **Origem:** `/Volumes/horizonte/bronze/landing/erp/`
- **Fonte:** dados sintéticos gerados por `data/geradores/gera_erp.py` (seed 42; premissas em D16 e D17)
- **Tabelas geradas:** `erp_lojas`, `erp_produtos`, `erp_vendas`, `erp_estoque_diario`

### Ingestão
Mesmo padrão da bronze do Olist (D05 e D06):
- **Tudo como texto** (`inferSchema => false`): a bronze guarda o dado exatamente como chegou. A tipagem acontece na silver.
- **Colunas de auditoria:** `_arquivo_origem` e `_ingerido_em`.
- **Carga completa** (`CREATE OR REPLACE`): os arquivos do gerador são estáticos.
- **Arquivos `.csv.gz` lidos direto:** o Spark reconhece a compressão pela extensão. O gzip não pode ser dividido entre tarefas, mas no volume do ERP (1,26 milhão de linhas) isso não pesa.
- **A sujeira plantada é mantida:** duplicadas, SKU nulo e quantidade negativa ficam na bronze. Quem marca é a silver (D07).

**Equivalente no BigQuery:** tabela externa sobre o bucket de landing com colunas `STRING` e `compression = 'GZIP'`; a origem vem de `_FILE_NAME` e a data de carga de `CURRENT_TIMESTAMP()`.

In [0]:
CREATE OR REPLACE TABLE horizonte.bronze.erp_lojas AS
SELECT * EXCEPT (_rescued_data),
       _metadata.file_name AS _arquivo_origem,
       current_timestamp() AS _ingerido_em
FROM read_files('/Volumes/horizonte/bronze/landing/erp/erp_lojas.csv',
                format => 'csv', header => true, inferSchema => false);

In [0]:
CREATE OR REPLACE TABLE horizonte.bronze.erp_produtos AS
SELECT * EXCEPT (_rescued_data),
       _metadata.file_name AS _arquivo_origem,
       current_timestamp() AS _ingerido_em
FROM read_files('/Volumes/horizonte/bronze/landing/erp/erp_produtos.csv',
                format => 'csv', header => true, inferSchema => false);

In [0]:
CREATE OR REPLACE TABLE horizonte.bronze.erp_vendas AS
SELECT * EXCEPT (_rescued_data),
       _metadata.file_name AS _arquivo_origem,
       current_timestamp() AS _ingerido_em
FROM read_files('/Volumes/horizonte/bronze/landing/erp/erp_vendas.csv.gz',
                format => 'csv', header => true, inferSchema => false);

In [0]:
CREATE OR REPLACE TABLE horizonte.bronze.erp_estoque_diario AS
SELECT * EXCEPT (_rescued_data),
       _metadata.file_name AS _arquivo_origem,
       current_timestamp() AS _ingerido_em
FROM read_files('/Volumes/horizonte/bronze/landing/erp/erp_estoque_diario.csv.gz',
                format => 'csv', header => true, inferSchema => false);

In [0]:
SELECT 'erp_lojas' AS tabela, COUNT(*) AS linhas FROM horizonte.bronze.erp_lojas      -- 12
UNION ALL SELECT 'erp_produtos', COUNT(*) FROM horizonte.bronze.erp_produtos             -- 200
UNION ALL SELECT 'erp_vendas', COUNT(*) FROM horizonte.bronze.erp_vendas                 -- 401.335
UNION ALL SELECT 'erp_estoque_diario', COUNT(*) FROM horizonte.bronze.erp_estoque_diario; -- 1.258.331

### A sujeira chegou intacta?
A bronze não pode corrigir nada. Os valores devem bater com os do gerador (D17); as marcações por linha ficam na silver.

In [0]:
SELECT
  COUNT_IF(sku IS NULL OR sku = '')    AS sku_nulo,              -- 400
  COUNT_IF(quantidade LIKE '-%')       AS quantidade_negativa,   -- 401
  COUNT(*) - (SELECT COUNT(*) FROM (
      SELECT DISTINCT cupom_id, data_hora, loja_id, sku, quantidade, preco_unitario, valor_total
      FROM horizonte.bronze.erp_vendas)) AS linhas_duplicadas    -- 801
FROM horizonte.bronze.erp_vendas;